# N-HiTS Forecasting: Complete Refactored Workflow
**Production-Grade Time Series Forecasting with Single Model Training, Hyperparameter Search & Easy Evaluation**

---

### Overview
This notebook provides:
- ✅ **Single model training** for chosen demand zone + months
- ✅ **Unique model naming & metadata** tracking for easy retrieval
- ✅ **Evaluation workflow** across different time periods and zones
- ✅ **Hyperparameter search** with grid exploration
- ✅ **Results comparison** and model ranking

### N-HiTS vs N-BEATS
N-HiTS (Neural Hierarchical Interpolation for Time Series) improves on N-BEATS by:
- **Hierarchical interpolation** — multi-rate signal sampling via pooling kernels
- **Reduced computation** — significantly faster training with comparable accuracy
- **Better long-horizon** — explicit frequency decomposition across stacks

### Directory Structure
```
./models/       → N-HiTS checkpoints + metadata JSON
./results/      → Evaluation plots, metrics CSVs
./logs/         → TensorBoard logs (1 per model)
```

## Cell 1: Header & Imports

In [ ]:
# ============================================================================
# N-HiTS Refactored Notebook - Imports & Setup
# ============================================================================

import subprocess
import sys

def install_if_needed(package, import_name=None):
    if import_name is None:
        import_name = package
    try:
        __import__(import_name)
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

install_if_needed("darts")
install_if_needed("pytorch_lightning")
install_if_needed("plotly")
install_if_needed("torch")

# Standard imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from pathlib import Path
import json
import os
from datetime import datetime
from typing import Dict, Tuple, Optional, List
import hashlib
import logging

# Darts imports
from darts import TimeSeries, concatenate
from darts.models import NHiTSModel
from darts.dataprocessing.transformers import Scaler, MissingValuesFiller
from pytorch_lightning.callbacks.early_stopping import EarlyStopping
from pytorch_lightning.loggers import TensorBoardLogger
from darts.utils.callbacks import TFMProgressBar
import torch
import joblib

# Suppress warnings
import warnings
warnings.filterwarnings("ignore")
logging.disable(logging.CRITICAL)

print("✅ All imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

## Cell 2: Configuration (USER EDITS HERE)

In [ ]:
# ============================================================================
# CONFIGURATION - Edit these values for your experiment
# ============================================================================

class Config:
    """Centralized configuration for all experiments."""
    
    # ========== Paths ==========
    DATA_FILE = "cleaned_demand_data.csv"
    MODELS_DIR = Path("./models")
    RESULTS_DIR = Path("./results")
    LOGS_DIR = Path("./logs")
    REGISTRY_FILE = MODELS_DIR / "model_registry.json"
    
    # Create directories if they don't exist
    MODELS_DIR.mkdir(exist_ok=True)
    RESULTS_DIR.mkdir(exist_ok=True)
    LOGS_DIR.mkdir(exist_ok=True)
    
    # ========== Data Configuration ==========
    TIME_COLUMN = "Timestamp"
    AVAILABLE_ZONES = [
        "TPCODL Demand",
        "TPWODL Demand",
        "TPNODL Demand",
        "TPSOSDL Demand",
        "Total Demand (as recorded)"
    ]
    
    # ========== Single Model Training Configuration ==========
    TRAINING_ZONE = "TPCODL Demand"
    TRAINING_MONTHS = [1, 11, 12]
    TRAINING_START_DATE = "2020-01-01"
    TRAINING_END_DATE = "2024-12-31"
    
    # Optional: batch month groups for overnight HPO runs
    HPO_TRAINING_MONTH_GROUPS = [
        [1, 11, 12],
        [4, 5, 6],
        [2, 3],
        [7, 8, 9, 10],
    ]
    
    # Train/val split
    TRAIN_VAL_SPLIT = 0.75
    
    # ========== N-HiTS Model Hyperparameters ==========
    INPUT_DAYS = 7
    INPUT_CHUNK_LENGTH = 96 * INPUT_DAYS + 18 * 4  # 15-min intervals = 744 steps
    OUTPUT_CHUNK_LENGTH = 24 * 4  # 1 day ahead (96 steps)
    OUTPUT_SHIFT = 6 * 4  # Shift parameter
    
    NUM_LAYERS = 2        # Layers per block
    LAYER_WIDTHS = 512    # Hidden units
    NUM_STACKS = 3        # N-HiTS typically uses fewer stacks (3 = fast/medium/slow)
    NUM_BLOCKS = 1        # Blocks per stack
    
    # N-HiTS specific: pooling kernel sizes per stack (controls hierarchical decomposition)
    # None = auto-computed by Darts based on input/output lengths
    POOLING_KERNEL_SIZES = None
    N_FREQ_DOWNSAMPLE = None  # None = auto
    MAXPOOL1D = True  # Use MaxPool1d (True) vs AvgPool1d (False)
    DROPOUT = 0.1
    
    # Training hyperparameters
    N_EPOCHS = 100
    BATCH_SIZE = 256
    NR_EPOCHS_VAL_PERIOD = 1
    RANDOM_STATE = 47
    
    # Early stopping
    EARLY_STOPPING_PATIENCE = 10
    EARLY_STOPPING_MIN_DELTA = 1e-5
    
    # ========== Hyperparameter Search Grid ==========
    HPO_GRID = {
        "input_chunk_length": [96 * i + 18 * 4 for i in [3, 5, 7, 10, 14]],
        "num_layers": [1, 2],
        "layer_widths": [256, 512, 1024],
        "num_stacks": [3],     # 3 stacks is standard for N-HiTS
        "batch_size": [256],
    }
    # Total combinations per month-group: 5 × 2 × 3 × 1 × 1 = 30
    
    # ========== Evaluation Configuration ==========
    TEST_START_DATE = "2025-01-01"
    TEST_END_DATE = "2026-01-14"
    EVAL_ZONE = "TPCODL Demand"
    EVAL_MONTHS = None  # None = all months
    
    @classmethod
    def print_config(cls):
        """Print active configuration."""
        print("\n" + "="*60)
        print("ACTIVE CONFIGURATION (N-HiTS)")
        print("="*60)
        print(f"Training Zone: {cls.TRAINING_ZONE}")
        print(f"Single Training Months: {cls.TRAINING_MONTHS}")
        print(f"HPO Month Groups: {cls.HPO_TRAINING_MONTH_GROUPS}")
        print(f"Input Context: {cls.INPUT_DAYS} days ({cls.INPUT_CHUNK_LENGTH} steps)")
        print(f"Model Arch: {cls.NUM_STACKS} stacks, {cls.NUM_LAYERS} layers, width={cls.LAYER_WIDTHS}")
        print(f"N-HiTS Specific: MaxPool1d={cls.MAXPOOL1D}, dropout={cls.DROPOUT}")
        print(f"Training: {cls.N_EPOCHS} epochs, batch_size={cls.BATCH_SIZE}")
        print("="*60 + "\n")

Config.print_config()

## Cell 3: Utility Classes

In [ ]:
# ============================================================================
# UTILITY CLASSES - Core functionality
# ============================================================================

class DataPreprocessor:
    """Handles data loading, filtering, scaling, and preprocessing."""
    
    def __init__(self):
        self.scaler = None
        self.filler = MissingValuesFiller()
    
    def load_data(self, filepath: str) -> pd.DataFrame:
        """Load CSV data."""
        df = pd.read_csv(filepath)
        df['Timestamp'] = pd.to_datetime(df['Timestamp'])
        if 'Total Demand (as recorded)' in df.columns:
            df.rename(columns={'Total Demand (as recorded)': 'Total Demand'}, inplace=True)
        return df
    
    def filter_by_zone_and_months(self, df: pd.DataFrame, zone: str,
                                   start_date: str, end_date: str,
                                   months: Optional[List[int]] = None) -> pd.DataFrame:
        """Filter dataframe by zone, date range, and months."""
        df_filtered = df.copy()
        df_filtered['Timestamp'] = pd.to_datetime(df_filtered['Timestamp'])
        df_filtered = df_filtered[
            (df_filtered['Timestamp'] >= start_date) &
            (df_filtered['Timestamp'] <= end_date)
        ]
        if months:
            df_filtered = df_filtered[df_filtered['Timestamp'].dt.month.isin(months)]
        return df_filtered[[Config.TIME_COLUMN, zone]].copy()
    
    def split_train_val(self, series: TimeSeries, split_ratio: float = 0.75) -> Tuple[TimeSeries, TimeSeries]:
        """Split series into train and validation."""
        split_index = int(len(series) * split_ratio)
        return series[:split_index], series[split_index:]
    
    def preprocess_series(self, df: pd.DataFrame, target_col: str, time_col: str = "Timestamp"):
        """Convert DataFrame to TimeSeries, fill missing values, and scale."""
        series = TimeSeries.from_dataframe(df, time_col=time_col, value_cols=[target_col], freq="15min")
        series = self.filler.transform(series)
        train, val = self.split_train_val(series, Config.TRAIN_VAL_SPLIT)
        self.scaler = Scaler()
        train = self.scaler.fit_transform(train)
        val = self.scaler.transform(val)
        return train, val, self.scaler
    
    def prepare_test_series(self, df: pd.DataFrame, target_col: str, scaler: Scaler,
                            time_col: str = "Timestamp") -> TimeSeries:
        """Prepare test series with same scaler."""
        series = TimeSeries.from_dataframe(df, time_col=time_col, value_cols=[target_col], freq="15min")
        series = self.filler.transform(series)
        series = scaler.transform(series)
        return series


class ModelBuilder:
    """Encapsulates N-HiTS model instantiation."""
    
    @staticmethod
    def build(model_name: str, input_chunk_length: int, num_layers: int,
             layer_widths: int, num_stacks: int, batch_size: int,
             early_stopper: EarlyStopping, logger: TensorBoardLogger) -> NHiTSModel:
        """Build N-HiTS model with specified hyperparameters."""
        
        torch.serialization.add_safe_globals([torch.optim.Adam])
        
        model = NHiTSModel(
            input_chunk_length=input_chunk_length,
            output_chunk_length=Config.OUTPUT_CHUNK_LENGTH,
            output_chunk_shift=Config.OUTPUT_SHIFT,
            num_stacks=num_stacks,
            num_blocks=Config.NUM_BLOCKS,
            num_layers=num_layers,
            layer_widths=layer_widths,
            pooling_kernel_sizes=Config.POOLING_KERNEL_SIZES,
            n_freq_downsample=Config.N_FREQ_DOWNSAMPLE,
            dropout=Config.DROPOUT,
            MaxPool1d=Config.MAXPOOL1D,
            n_epochs=Config.N_EPOCHS,
            nr_epochs_val_period=Config.NR_EPOCHS_VAL_PERIOD,
            batch_size=batch_size,
            random_state=Config.RANDOM_STATE,
            model_name=model_name,
            save_checkpoints=True,
            force_reset=True,
            pl_trainer_kwargs={
                "logger": logger,
                "accelerator": "gpu" if torch.cuda.is_available() else "cpu",
                "callbacks": [
                    early_stopper,
                    TFMProgressBar(enable_train_bar_only=True)
                ]
            }
        )
        
        return model


class ModelManager:
    """Manages model naming, persistence, and metadata tracking."""
    
    @staticmethod
    def generate_model_name(zone: str, config_dict: Dict) -> str:
        """Generate unique model name from zone and config."""
        zone_code = zone.split()[0]
        sig = (
            f"{config_dict.get('num_layers', 2)}l_"
            f"{config_dict.get('layer_widths', 512)}w_"
            f"ctx{config_dict.get('input_chunk_length', 744) // 96}d"
        )
        timestamp = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
        return f"nhits_{zone_code}_{sig}_{timestamp}"
    
    @staticmethod
    def save_model_metadata(model_name: str, zone: str, config: Dict,
                           metrics: Dict, train_time_sec: float):
        """Save model metadata to registry."""
        registry = {}
        if Config.REGISTRY_FILE.exists():
            with open(Config.REGISTRY_FILE, 'r') as f:
                registry = json.load(f)
        registry[model_name] = {
            "zone": zone,
            "config": config,
            "metrics": metrics,
            "train_time_sec": train_time_sec,
            "checkpoint_dir": str(Config.MODELS_DIR / model_name),
            "created_at": datetime.utcnow().isoformat(),
            "scaler_path": str(Config.MODELS_DIR / model_name / "scaler.joblib")
        }
        with open(Config.REGISTRY_FILE, 'w') as f:
            json.dump(registry, f, indent=2)
    
    @staticmethod
    def load_registry() -> Dict:
        """Load model registry."""
        if not Config.REGISTRY_FILE.exists():
            return {}
        with open(Config.REGISTRY_FILE, 'r') as f:
            return json.load(f)
    
    @staticmethod
    def list_models(zone: Optional[str] = None) -> List[str]:
        """List all trained models, optionally filtered by zone."""
        registry = ModelManager.load_registry()
        if zone:
            return [name for name, entry in registry.items() if entry.get("zone") == zone]
        return list(registry.keys())


class ModelEvaluator:
    """Computes metrics and generates evaluation visualizations."""
    
    @staticmethod
    def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, name: str = "Model") -> Dict:
        """Compute standard regression metrics."""
        errors = y_pred - y_true
        mae = np.mean(np.abs(errors))
        mse = np.mean(errors ** 2)
        rmse = np.sqrt(mse)
        mape = np.mean(np.abs(errors / (np.abs(y_true) + 1e-8))) * 100
        mae_pct = (mae / (np.abs(y_true).mean() + 1e-8)) * 100
        r2 = 1 - (np.sum(errors**2) / np.sum((y_true - y_true.mean())**2 + 1e-8))
        return {"MAE": mae, "RMSE": rmse, "MAPE": mape, "MAE%": mae_pct, "R2": r2, "Errors": errors}
    
    @staticmethod
    def print_metrics(metrics: Dict, name: str = "Model"):
        """Pretty print metrics."""
        print(f"\n--- {name} Metrics ---")
        print(f"MAE:    {metrics['MAE']:.4f}")
        print(f"RMSE:   {metrics['RMSE']:.4f}")
        print(f"MAPE:   {metrics['MAPE']:.2f}%")
        print(f"MAE%:   {metrics['MAE%']:.2f}%")
        print(f"R²:     {metrics['R2']:.4f}")


class HPORunner:
    """Orchestrates hyperparameter search across grid with test evaluation."""

    @staticmethod
    def evaluate_on_test_set(model, zone, preprocessor_obj=None, df_full_data=None, scaler_obj=None) -> Dict:
        """Evaluate a trained model on the configured test window."""
        empty_metrics = {"test_MAPE": np.nan, "test_MAE": np.nan, "test_R2": np.nan}
        if preprocessor_obj is None or df_full_data is None or scaler_obj is None:
            return empty_metrics

        df_test = preprocessor_obj.filter_by_zone_and_months(
            df_full_data, zone, Config.TEST_START_DATE, Config.TEST_END_DATE, Config.EVAL_MONTHS
        )
        if df_test.empty:
            return empty_metrics

        test_series_scaled = preprocessor_obj.prepare_test_series(df_test, zone, scaler_obj)
        predictions_scaled = model.historical_forecasts(
            test_series_scaled,
            forecast_horizon=Config.OUTPUT_CHUNK_LENGTH,
            stride=Config.OUTPUT_CHUNK_LENGTH,
            last_points_only=False, retrain=False, verbose=False,
        )
        if not predictions_scaled:
            return empty_metrics

        predictions_scaled = concatenate(predictions_scaled)
        predictions_unscaled = scaler_obj.inverse_transform(predictions_scaled)
        predictions_df = predictions_unscaled.to_dataframe().reset_index()
        if predictions_df.shape[1] < 2:
            return empty_metrics

        predictions_df = predictions_df.rename(
            columns={predictions_df.columns[0]: "Timestamp", predictions_df.columns[1]: "Predicted"}
        )
        predictions_df = predictions_df[["Timestamp", "Predicted"]]
        predictions_df["Timestamp"] = pd.to_datetime(predictions_df["Timestamp"])

        df_actual = df_test[[Config.TIME_COLUMN, zone]].copy()
        df_actual.columns = ["Timestamp", "Actual"]
        df_actual["Timestamp"] = pd.to_datetime(df_actual["Timestamp"])

        df_eval = pd.merge(df_actual, predictions_df, on="Timestamp", how="inner").dropna(subset=["Actual", "Predicted"])
        if df_eval.empty:
            return empty_metrics

        metrics_eval = ModelEvaluator.compute_metrics(df_eval["Actual"].values, df_eval["Predicted"].values, zone)
        return {"test_MAPE": metrics_eval["MAPE"], "test_MAE": metrics_eval["MAE"], "test_R2": metrics_eval["R2"]}

    @staticmethod
    def run_grid_search(train, val, zone, grid, save_results=True,
                        preprocessor_obj=None, df_full_data=None, scaler_base=None):
        """Run grid search over hyperparameter combinations."""
        results = []
        model_names = []
        import itertools

        keys = list(grid.keys())
        values = [grid[k] for k in keys]
        combinations = list(itertools.product(*values))

        print(f"\n{'=' * 70}")
        print(f"Starting HPO: {len(combinations)} combinations to test")
        print(f"{'=' * 70}\n")

        for i, combo in enumerate(combinations, 1):
            config = dict(zip(keys, combo))
            print(f"\n[{i}/{len(combinations)}] Config: {config}")
            print("-" * 70)

            model_name = ModelManager.generate_model_name(zone, config)
            model_names.append(model_name)

            logger = TensorBoardLogger(save_dir=str(Config.LOGS_DIR), name=model_name)
            early_stopper = EarlyStopping(
                monitor="val_loss",
                patience=Config.EARLY_STOPPING_PATIENCE,
                min_delta=Config.EARLY_STOPPING_MIN_DELTA,
                mode="min",
            )

            try:
                start_time = datetime.now()
                model = ModelBuilder.build(
                    model_name, config["input_chunk_length"], config["num_layers"],
                    config["layer_widths"], config["num_stacks"], config["batch_size"],
                    early_stopper, logger,
                )
                model.fit(train, val_series=val, verbose=False)

                torch.serialization.add_safe_globals([torch.optim.Adam])
                model = NHiTSModel.load_from_checkpoint(model_name=model_name, best=True)
                train_time = (datetime.now() - start_time).total_seconds()

                model_dir = Config.MODELS_DIR / model_name
                model_dir.mkdir(exist_ok=True)
                if scaler_base is not None:
                    joblib.dump(scaler_base, model_dir / "scaler.joblib")

                test_metrics = HPORunner.evaluate_on_test_set(
                    model, zone,
                    preprocessor_obj=preprocessor_obj,
                    df_full_data=df_full_data,
                    scaler_obj=scaler_base,
                )
                metrics = {"train_time_sec": train_time, **test_metrics}
                ModelManager.save_model_metadata(model_name, zone, config, metrics, train_time)

                results.append({
                    "model_name": model_name, "config": config,
                    "train_time_sec": train_time, **test_metrics, "status": "✅ Success",
                })

                if not np.isnan(test_metrics["test_MAPE"]):
                    print(f"   Test -> MAPE: {test_metrics['test_MAPE']:.2f}% | MAE: {test_metrics['test_MAE']:.2f} | R2: {test_metrics['test_R2']:.4f}")
                else:
                    print("   Test metrics unavailable")
                print(f"✅ Trained in {train_time:.1f}s - Checkpoint: {model_name}")

            except Exception as e:
                print(f"❌ Failed: {str(e)[:120]}")
                results.append({
                    "model_name": model_name, "config": config,
                    "error": str(e), "test_MAPE": np.nan, "test_MAE": np.nan,
                    "test_R2": np.nan, "status": "❌ Failed",
                })

        results_df = pd.DataFrame(results)
        print(f"\n{'=' * 70}")
        print(f"HPO Complete: {len([r for r in results if r['status'].startswith('✅')])} / {len(combinations)} successful")
        print(f"{'=' * 70}\n")

        if save_results:
            results_csv = Config.RESULTS_DIR / f"nhits_hpo_results_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}.csv"
            results_df.to_csv(results_csv, index=False)
            print(f"Results saved to: {results_csv}")

        return results_df, model_names


print("✅ All utility classes loaded successfully!")

In [ ]:
# ============================================================================
# UTILITY OVERRIDES - Training-data metadata support
# ============================================================================

def _normalize_month_group(months: Optional[List[int]]) -> Optional[List[int]]:
    if months is None:
        return None
    normalized = []
    seen = set()
    for month in months:
        month_int = int(month)
        if month_int not in seen:
            normalized.append(month_int)
            seen.add(month_int)
    return normalized


def _month_group_label(months: Optional[List[int]]) -> str:
    if months is None:
        return "all_months"
    return "months_" + "_".join(str(month) for month in months)


def _generate_model_name(zone: str, config_dict: Dict) -> str:
    zone_code = zone.split()[0]
    sig = (
        f"{config_dict.get('num_layers', 2)}l_"
        f"{config_dict.get('layer_widths', 512)}w_"
        f"ctx{config_dict.get('input_chunk_length', 744) // 96}d"
    )
    requested_months = config_dict.get("requested_months")
    month_label = config_dict.get("requested_months_label")
    if month_label is None and requested_months is not None:
        month_label = _month_group_label(_normalize_month_group(requested_months))
    timestamp = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
    if month_label:
        return f"nhits_{zone_code}_{month_label}_{sig}_{timestamp}"
    return f"nhits_{zone_code}_{sig}_{timestamp}"


def _build_training_data_metadata(
    df_filtered: pd.DataFrame, zone: str,
    requested_start_date: str, requested_end_date: str,
    requested_months: Optional[List[int]], split_ratio: float,
) -> Dict:
    normalized_months = _normalize_month_group(requested_months)
    if df_filtered.empty:
        return {
            "zone": zone, "requested_start_date": requested_start_date,
            "requested_end_date": requested_end_date,
            "requested_months": normalized_months,
            "requested_months_label": _month_group_label(normalized_months),
            "actual_start_timestamp": None, "actual_end_timestamp": None,
            "training_years": [], "training_months_present": [],
            "n_rows": 0, "train_val_split": split_ratio,
        }
    timestamps = pd.to_datetime(df_filtered[Config.TIME_COLUMN])
    return {
        "zone": zone, "requested_start_date": requested_start_date,
        "requested_end_date": requested_end_date,
        "requested_months": normalized_months,
        "requested_months_label": _month_group_label(normalized_months),
        "actual_start_timestamp": timestamps.min().isoformat(),
        "actual_end_timestamp": timestamps.max().isoformat(),
        "training_years": sorted(int(y) for y in timestamps.dt.year.unique()),
        "training_months_present": sorted(int(m) for m in timestamps.dt.month.unique()),
        "n_rows": int(len(df_filtered)), "train_val_split": split_ratio,
    }


def _save_model_metadata(
    model_name: str, zone: str, config: Dict, metrics: Dict,
    train_time_sec: float, training_data: Optional[Dict] = None,
):
    registry = {}
    if Config.REGISTRY_FILE.exists():
        with open(Config.REGISTRY_FILE, 'r') as f:
            registry = json.load(f)
    entry = {
        "zone": zone, "config": config, "metrics": metrics,
        "train_time_sec": train_time_sec,
        "checkpoint_dir": str(Config.MODELS_DIR / model_name),
        "created_at": datetime.utcnow().isoformat(),
        "scaler_path": str(Config.MODELS_DIR / model_name / "scaler.joblib")
    }
    if training_data is not None:
        entry["training_data"] = training_data
        entry["training_years"] = training_data.get("training_years", [])
        entry["training_months_present"] = training_data.get("training_months_present", [])
        entry["requested_months"] = training_data.get("requested_months")
        entry["requested_months_label"] = training_data.get("requested_months_label")
        entry["requested_start_date"] = training_data.get("requested_start_date")
        entry["requested_end_date"] = training_data.get("requested_end_date")
    registry[model_name] = entry
    with open(Config.REGISTRY_FILE, 'w') as f:
        json.dump(registry, f, indent=2)


def _run_grid_search(
    train, val, zone, grid, save_results=True,
    preprocessor_obj=None, df_full_data=None, scaler_base=None,
    training_data_metadata=None,
):
    results = []
    model_names = []
    import itertools

    keys = list(grid.keys())
    values = [grid[k] for k in keys]
    combinations = list(itertools.product(*values))

    print(f"\n{'=' * 70}")
    print(f"Starting N-HiTS HPO: {len(combinations)} combinations")
    print(f"{'=' * 70}\n")

    for i, combo in enumerate(combinations, 1):
        config = dict(zip(keys, combo))
        print(f"\n[{i}/{len(combinations)}] Config: {config}")
        print("-" * 70)

        naming_config = dict(config)
        if training_data_metadata is not None:
            naming_config["requested_months"] = training_data_metadata.get("requested_months")
            naming_config["requested_months_label"] = training_data_metadata.get("requested_months_label")

        model_name = ModelManager.generate_model_name(zone, naming_config)
        model_names.append(model_name)

        logger = TensorBoardLogger(save_dir=str(Config.LOGS_DIR), name=model_name)
        early_stopper = EarlyStopping(
            monitor="val_loss", patience=Config.EARLY_STOPPING_PATIENCE,
            min_delta=Config.EARLY_STOPPING_MIN_DELTA, mode="min",
        )

        try:
            start_time = datetime.now()
            model = ModelBuilder.build(
                model_name, config["input_chunk_length"], config["num_layers"],
                config["layer_widths"], config["num_stacks"], config["batch_size"],
                early_stopper, logger,
            )
            model.fit(train, val_series=val, verbose=False)

            torch.serialization.add_safe_globals([torch.optim.Adam])
            model = NHiTSModel.load_from_checkpoint(model_name=model_name, best=True)
            train_time = (datetime.now() - start_time).total_seconds()

            model_dir = Config.MODELS_DIR / model_name
            model_dir.mkdir(exist_ok=True)
            if scaler_base is not None:
                joblib.dump(scaler_base, model_dir / "scaler.joblib")

            test_metrics = HPORunner.evaluate_on_test_set(
                model, zone,
                preprocessor_obj=preprocessor_obj,
                df_full_data=df_full_data,
                scaler_obj=scaler_base,
            )
            metrics = {"train_time_sec": train_time, **test_metrics}
            ModelManager.save_model_metadata(
                model_name, zone, config, metrics, train_time,
                training_data=training_data_metadata,
            )

            results.append({
                "model_name": model_name, "config": config,
                "train_time_sec": train_time, **test_metrics, "status": "✅ Success",
            })

            if not np.isnan(test_metrics["test_MAPE"]):
                print(f"   Test -> MAPE: {test_metrics['test_MAPE']:.2f}% | MAE: {test_metrics['test_MAE']:.2f} | R2: {test_metrics['test_R2']:.4f}")
            else:
                print("   Test metrics unavailable")
            print(f"✅ Trained in {train_time:.1f}s - Checkpoint: {model_name}")

        except Exception as e:
            print(f"❌ Failed: {str(e)[:120]}")
            results.append({
                "model_name": model_name, "config": config,
                "error": str(e), "test_MAPE": np.nan, "test_MAE": np.nan,
                "test_R2": np.nan, "status": "❌ Failed",
            })

    results_df = pd.DataFrame(results)
    print(f"\n{'=' * 70}")
    print(f"HPO Complete: {len([r for r in results if r['status'].startswith('✅')])} / {len(combinations)} successful")
    print(f"{'=' * 70}\n")

    if training_data_metadata is not None and not results_df.empty:
        results_df["requested_months"] = [training_data_metadata.get("requested_months")] * len(results_df)
        results_df["requested_months_label"] = [training_data_metadata.get("requested_months_label")] * len(results_df)
        results_df["training_years"] = [training_data_metadata.get("training_years", [])] * len(results_df)
        results_df["training_months_present"] = [training_data_metadata.get("training_months_present", [])] * len(results_df)

    if save_results:
        results_csv = Config.RESULTS_DIR / f"nhits_hpo_results_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}.csv"
        results_df.to_csv(results_csv, index=False)
        print(f"Results saved to: {results_csv}")

    return results_df, model_names


def _run_month_group_grid_search(
    preprocessor_obj, df_full_data, zone, grid,
    month_groups, start_date, end_date, split_ratio, save_results=True,
):
    month_groups = month_groups or [Config.TRAINING_MONTHS]
    all_results = []
    all_model_names = []
    import itertools

    total_combinations = len(list(itertools.product(*[grid[k] for k in grid.keys()])))
    print("\n" + "=" * 80)
    print(f"Batch N-HiTS HPO across {len(month_groups)} month-group(s)")
    print(f"Models per month-group: {total_combinations}")
    print(f"Expected total models: {len(month_groups) * total_combinations}")
    print("=" * 80)

    for group_index, months in enumerate(month_groups, 1):
        normalized_months = _normalize_month_group(months)
        month_label = _month_group_label(normalized_months)
        print(f"\n[{group_index}/{len(month_groups)}] Preparing month-group: {normalized_months}")

        df_train_group = preprocessor_obj.filter_by_zone_and_months(
            df_full_data, zone, start_date, end_date, normalized_months,
        )
        training_data_metadata = ModelManager.build_training_data_metadata(
            df_train_group, zone, start_date, end_date, normalized_months, split_ratio,
        )
        if df_train_group.empty:
            print(f"   Skipping {month_label}: no rows found")
            continue

        print(f"   Rows: {len(df_train_group)}")
        print(f"   Years: {training_data_metadata['training_years']}")
        print(f"   Months present: {training_data_metadata['training_months_present']}")

        train_group, val_group, scaler_group = preprocessor_obj.preprocess_series(df_train_group, zone)

        group_results_df, group_model_names = HPORunner.run_grid_search(
            train_group, val_group, zone, grid, save_results=False,
            preprocessor_obj=preprocessor_obj, df_full_data=df_full_data,
            scaler_base=scaler_group, training_data_metadata=training_data_metadata,
        )
        if not group_results_df.empty:
            group_results_df["month_group_index"] = group_index
            group_results_df["requested_months"] = [training_data_metadata.get("requested_months")] * len(group_results_df)
            group_results_df["requested_months_label"] = [month_label] * len(group_results_df)
            group_results_df["training_years"] = [training_data_metadata.get("training_years", [])] * len(group_results_df)
            group_results_df["training_months_present"] = [training_data_metadata.get("training_months_present", [])] * len(group_results_df)
            all_results.append(group_results_df)
        all_model_names.extend(group_model_names)

    combined_results_df = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()
    if save_results and not combined_results_df.empty:
        results_csv = Config.RESULTS_DIR / f"nhits_hpo_month_groups_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}.csv"
        combined_results_df.to_csv(results_csv, index=False)
        print(f"\nCombined month-group results saved to: {results_csv}")
    return combined_results_df, all_model_names


# Apply overrides
ModelManager.generate_model_name = staticmethod(_generate_model_name)
ModelManager.build_training_data_metadata = staticmethod(_build_training_data_metadata)
ModelManager.save_model_metadata = staticmethod(_save_model_metadata)
HPORunner.run_grid_search = staticmethod(_run_grid_search)
HPORunner.run_month_group_grid_search = staticmethod(_run_month_group_grid_search)

print("✅ Utility overrides loaded: month-group labels included in model names")

## Cell 4: Data Loading & EDA

In [ ]:
# ============================================================================
# DATA LOADING & EXPLORATORY DATA ANALYSIS
# ============================================================================

preprocessor = DataPreprocessor()
df_full = preprocessor.load_data(Config.DATA_FILE)

print(f"✅ Data loaded: {df_full.shape}")
print(f"\nDate range: {df_full['Timestamp'].min()} to {df_full['Timestamp'].max()}")
print(f"\nAvailable columns: {list(df_full.columns)}")
print(f"\nBasic stats for {Config.TRAINING_ZONE}:")
print(df_full[Config.TRAINING_ZONE].describe())

In [ ]:
# Visualize training zone demand pattern
df_plot = df_full.copy()
df_plot = df_plot.set_index('Timestamp').resample('1H').mean().reset_index()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_plot['Timestamp'],
    y=df_plot[Config.TRAINING_ZONE],
    mode='lines', name=Config.TRAINING_ZONE,
    line=dict(color='steelblue', width=1)
))
fig.update_layout(
    title=f"Demand Pattern: {Config.TRAINING_ZONE}",
    xaxis_title="Timestamp", yaxis_title="Demand",
    hovermode='x unified', height=400
)
fig.show()
print("✅ Visualization complete")

## Cell 5: Single Model Training

In [ ]:
# ============================================================================
# SINGLE MODEL TRAINING
# ============================================================================

print("\n" + "="*70)
print("SINGLE N-HiTS MODEL TRAINING")
print("="*70)

# Step 1: Filter data
print(f"\n[1/4] Filtering data for {Config.TRAINING_ZONE}...")
df_train_raw = preprocessor.filter_by_zone_and_months(
    df_full, Config.TRAINING_ZONE,
    Config.TRAINING_START_DATE, Config.TRAINING_END_DATE,
    Config.TRAINING_MONTHS
)
print(f"   Filtered shape: {df_train_raw.shape}")
print(f"   Date range: {df_train_raw['Timestamp'].min()} to {df_train_raw['Timestamp'].max()}")

training_data_metadata = ModelManager.build_training_data_metadata(
    df_train_raw, Config.TRAINING_ZONE,
    Config.TRAINING_START_DATE, Config.TRAINING_END_DATE,
    Config.TRAINING_MONTHS, Config.TRAIN_VAL_SPLIT,
)
print(f"   Training years: {training_data_metadata['training_years']}")
print(f"   Months present: {training_data_metadata['training_months_present']}")
print(f"   Month label: {training_data_metadata['requested_months_label']}")

# Step 2: Preprocess
print(f"\n[2/4] Preprocessing...")
train_series, val_series, scaler = preprocessor.preprocess_series(df_train_raw, Config.TRAINING_ZONE)
print(f"   Train series: {train_series.shape}")
print(f"   Val series: {val_series.shape}")

# Step 3: Generate model name
print(f"\n[3/4] Generating model name...")
config_dict = {
    "input_chunk_length": Config.INPUT_CHUNK_LENGTH,
    "num_layers": Config.NUM_LAYERS,
    "layer_widths": Config.LAYER_WIDTHS,
    "num_stacks": Config.NUM_STACKS,
    "batch_size": Config.BATCH_SIZE,
    "requested_months": training_data_metadata["requested_months"],
    "requested_months_label": training_data_metadata["requested_months_label"],
}
model_name_single = ModelManager.generate_model_name(Config.TRAINING_ZONE, config_dict)
print(f"   Model name: {model_name_single}")

# Step 4: Train
print(f"\n[4/4] Training N-HiTS model...")
print(f"   Input: {Config.INPUT_CHUNK_LENGTH} steps ({Config.INPUT_DAYS} days)")
print(f"   Output: {Config.OUTPUT_CHUNK_LENGTH} steps")
print(f"   Architecture: {Config.NUM_STACKS} stacks, {Config.NUM_LAYERS} layers, width={Config.LAYER_WIDTHS}")
print(f"   N-HiTS: MaxPool={Config.MAXPOOL1D}, dropout={Config.DROPOUT}")

start_time = datetime.now()

logger_single = TensorBoardLogger(save_dir=str(Config.LOGS_DIR), name=model_name_single)
early_stopper_single = EarlyStopping(
    monitor="val_loss", patience=Config.EARLY_STOPPING_PATIENCE,
    min_delta=Config.EARLY_STOPPING_MIN_DELTA, mode="min"
)

model_single = ModelBuilder.build(
    model_name_single, Config.INPUT_CHUNK_LENGTH, Config.NUM_LAYERS,
    Config.LAYER_WIDTHS, Config.NUM_STACKS, Config.BATCH_SIZE,
    early_stopper_single, logger_single
)

model_single.fit(train_series, val_series=val_series, verbose=False)
train_time = (datetime.now() - start_time).total_seconds()

# Load best checkpoint
torch.serialization.add_safe_globals([torch.optim.Adam])
model_single = NHiTSModel.load_from_checkpoint(model_name=model_name_single, best=True)

# Save scaler
model_dir = Config.MODELS_DIR / model_name_single
model_dir.mkdir(exist_ok=True)
joblib.dump(scaler, model_dir / "scaler.joblib")

# Save metadata
ModelManager.save_model_metadata(
    model_name_single, Config.TRAINING_ZONE, config_dict,
    {"train_time_sec": train_time}, train_time,
    training_data=training_data_metadata,
)

print(f"\n✅ Training complete in {train_time:.1f} seconds")
print(f"✅ Model saved: {model_name_single}")
print(f"✅ Checkpoint location: {model_dir}")
print(f"   Saved training years: {training_data_metadata['training_years']}")
print(f"   Saved training months: {training_data_metadata['training_months_present']}")
print(f"\nUse this model_name for evaluation: {model_name_single}")

## Cell 6: Evaluation on Test Set

In [ ]:
# ============================================================================
# EVALUATION ON TEST SET
# ============================================================================

print("\n" + "=" * 70)
print("N-HiTS MODEL EVALUATION WORKFLOW")
print("=" * 70)

print(f"\n[1/4] Loading model: {model_name_single}...")
model_eval = NHiTSModel.load_from_checkpoint(model_name=model_name_single, best=True)
scaler_eval = joblib.load(Config.MODELS_DIR / model_name_single / "scaler.joblib")
print("   Model loaded")

print(f"\n[2/4] Preparing test data ({Config.TEST_START_DATE} to {Config.TEST_END_DATE})...")
df_test = preprocessor.filter_by_zone_and_months(
    df_full, Config.TRAINING_ZONE,
    Config.TEST_START_DATE, Config.TEST_END_DATE, Config.EVAL_MONTHS
)
print(f"   Test shape: {df_test.shape}")

print("\n[3/4] Generating predictions...")
test_series_scaled = preprocessor.prepare_test_series(df_test, Config.TRAINING_ZONE, scaler_eval)

predictions_scaled = model_eval.historical_forecasts(
    test_series_scaled,
    forecast_horizon=Config.OUTPUT_CHUNK_LENGTH,
    stride=Config.OUTPUT_CHUNK_LENGTH,
    last_points_only=False, retrain=False, verbose=False
)

predictions_df = pd.DataFrame(columns=["Timestamp", "Predicted"])
if predictions_scaled:
    predictions_scaled = concatenate(predictions_scaled)
    predictions_unscaled = scaler_eval.inverse_transform(predictions_scaled)
    predictions_df = predictions_unscaled.to_dataframe().reset_index()
    if predictions_df.shape[1] >= 2:
        predictions_df = predictions_df.rename(
            columns={predictions_df.columns[0]: "Timestamp", predictions_df.columns[1]: "Predicted"}
        )
        predictions_df = predictions_df[["Timestamp", "Predicted"]]
        predictions_df["Timestamp"] = pd.to_datetime(predictions_df["Timestamp"])
    else:
        print("   No usable prediction columns")
else:
    print("   No predictions generated")
print(f"   Predictions shape: {predictions_df.shape}")

print("\n[4/4] Computing metrics...")
df_actual = df_test[["Timestamp", Config.TRAINING_ZONE]].copy()
df_actual.columns = ["Timestamp", "Actual"]
df_actual["Timestamp"] = pd.to_datetime(df_actual["Timestamp"])

df_eval = pd.merge(df_actual, predictions_df, on="Timestamp", how="inner").dropna(subset=["Actual", "Predicted"])

if len(df_eval) > 0:
    actual_vals = df_eval["Actual"].values
    pred_vals = df_eval["Predicted"].values
    metrics_eval = ModelEvaluator.compute_metrics(actual_vals, pred_vals, model_name_single)
    ModelEvaluator.print_metrics(metrics_eval, model_name_single)
    print(f"\n   Evaluation date range: {df_eval['Timestamp'].min()} to {df_eval['Timestamp'].max()}")
    print(f"   Samples evaluated: {len(df_eval)}")
else:
    print("   No overlapping data for evaluation")

print("\n✅ Evaluation complete")

In [ ]:
# Visualize predictions vs actual
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=df_eval['Timestamp'], y=df_eval['Actual'],
    mode='lines', name='Actual', line=dict(color='steelblue', width=2)
))
fig.add_trace(go.Scatter(
    x=df_eval['Timestamp'], y=df_eval['Predicted'],
    mode='lines', name='Predicted', line=dict(color='red', dash='dash', width=2)
))
fig.update_layout(
    title=f"Actual vs Predicted (N-HiTS) - {model_name_single}",
    xaxis_title="Timestamp", yaxis_title="Demand",
    hovermode='x unified', height=500
)
fig.show()

In [ ]:
# Plot error distribution
fig_error = go.Figure()
fig_error.add_trace(go.Scatter(
    x=df_eval['Timestamp'], y=metrics_eval['Errors'],
    mode='lines', name='Prediction Error', line=dict(color='orangered', width=1)
))
fig_error.update_layout(
    title=f"Prediction Error Over Time (N-HiTS) - {model_name_single}",
    xaxis_title="Timestamp", yaxis_title="Error (Predicted - Actual)",
    hovermode='x unified', height=400
)
fig_error.show()

## Cell 7: Hyperparameter Search

In [ ]:
# ============================================================================
# HYPERPARAMETER SEARCH (GRID SEARCH) - N-HiTS
# ============================================================================

month_groups = Config.HPO_TRAINING_MONTH_GROUPS

if month_groups:
    print("\nRunning batch N-HiTS HPO across configured month-groups")
    print(f"Month-groups: {month_groups}")

    hpo_results_df, hpo_model_names = HPORunner.run_month_group_grid_search(
        preprocessor_obj=preprocessor,
        df_full_data=df_full,
        zone=Config.TRAINING_ZONE,
        grid=Config.HPO_GRID,
        month_groups=month_groups,
        start_date=Config.TRAINING_START_DATE,
        end_date=Config.TRAINING_END_DATE,
        split_ratio=Config.TRAIN_VAL_SPLIT,
        save_results=True,
    )
else:
    print(f"\nUsing train/val series from single model training")
    print(f"  Train: {train_series.shape}, Val: {val_series.shape}")

    hpo_results_df, hpo_model_names = HPORunner.run_grid_search(
        train_series, val_series, Config.TRAINING_ZONE, Config.HPO_GRID,
        preprocessor_obj=preprocessor, df_full_data=df_full,
        scaler_base=scaler, training_data_metadata=training_data_metadata,
    )

print(f"\n✅ N-HiTS grid search complete")
print(f"Total models processed: {len(hpo_model_names)}")
print(f"\nTop Results:")

if hpo_results_df.empty:
    print("No HPO results were generated.")
else:
    display_columns = ["model_name", "requested_months", "test_MAPE", "test_MAE", "test_R2", "status"]
    available_columns = [col for col in display_columns if col in hpo_results_df.columns]
    print(hpo_results_df[available_columns].to_string(index=False))

## Cell 8: Results Analysis & Model Selection

In [ ]:
# ============================================================================
# RESULTS ANALYSIS & MODEL RANKING
# ============================================================================

print("\n" + "="*70)
print("N-HiTS MODEL REGISTRY & RANKING")
print("="*70)

registry = ModelManager.load_registry()
print(f"\nTotal models in registry: {len(registry)}")

# Filter only N-HiTS models for this zone
zone_models = {
    k: v for k, v in registry.items()
    if v.get("zone") == Config.TRAINING_ZONE and k.startswith("nhits_")
}
print(f"N-HiTS models for {Config.TRAINING_ZONE}: {len(zone_models)}")

if zone_models:
    ranking_data = []
    for model_name, metadata in zone_models.items():
        ranking_data.append({
            "model_name": model_name,
            "train_time_sec": metadata.get("train_time_sec", 0),
            "training_years": metadata.get("training_years", []),
            "requested_months": metadata.get("requested_months"),
            "training_months_present": metadata.get("training_months_present", []),
            "test_MAPE": metadata.get("metrics", {}).get("test_MAPE", np.nan),
            "test_R2": metadata.get("metrics", {}).get("test_R2", np.nan),
            "config": str(metadata.get("config", {}))
        })
    ranking_df = pd.DataFrame(ranking_data).sort_values("test_MAPE", ascending=True, na_position="last")
    print(f"\n{ranking_df.to_string(index=False)}")
    print(f"\n✅ {len(zone_models)} N-HiTS models available")
else:
    print(f"   No N-HiTS models trained yet for {Config.TRAINING_ZONE}")

print(f"\n✅ Registry saved to: {Config.REGISTRY_FILE}")

## Cell 9: Model Reload & Batch Evaluation

In [ ]:
# ============================================================================
# MODEL RELOAD + MULTI-MODEL PREDICTION PLOTTING (N-HiTS)
# ============================================================================

def list_available_models(zone=None, prefix="nhits_"):
    """Return model names from registry, filtered by zone and prefix."""
    registry_local = ModelManager.load_registry()
    models = []
    for name, m in registry_local.items():
        if prefix and not name.startswith(prefix):
            continue
        if zone is not None and m.get("zone") != zone:
            continue
        models.append(name)
    return models


def get_top_models_by_test_mape(zone=None, top_n=5, prefix="nhits_"):
    registry_local = ModelManager.load_registry()
    rows = []
    for model_name, meta in registry_local.items():
        if prefix and not model_name.startswith(prefix):
            continue
        if zone is not None and meta.get("zone") != zone:
            continue
        m = meta.get("metrics", {})
        rows.append({"model_name": model_name, "test_MAPE": m.get("test_MAPE", np.nan)})
    if not rows:
        return []
    df_rank = pd.DataFrame(rows).sort_values("test_MAPE", ascending=True, na_position="last")
    return df_rank["model_name"].head(top_n).tolist()


def _load_model_and_scaler(model_name):
    torch.serialization.add_safe_globals([torch.optim.Adam])
    model_obj = NHiTSModel.load_from_checkpoint(model_name=model_name, best=True)
    scaler_path = Config.MODELS_DIR / model_name / "scaler.joblib"
    if not scaler_path.exists():
        raise FileNotFoundError(f"Scaler not found for model: {model_name}")
    scaler_obj = joblib.load(scaler_path)
    return model_obj, scaler_obj


def _prediction_df_for_model(model_name, zone=None, start_date=None, end_date=None, months=None):
    if zone is None: zone = Config.TRAINING_ZONE
    if start_date is None: start_date = Config.TEST_START_DATE
    if end_date is None: end_date = Config.TEST_END_DATE
    if months is None: months = Config.EVAL_MONTHS

    model_obj, scaler_obj = _load_model_and_scaler(model_name)
    df_test_local = preprocessor.filter_by_zone_and_months(df_full, zone, start_date, end_date, months)
    if df_test_local.empty:
        raise ValueError(f"No test rows for model: {model_name}")

    test_series_scaled = preprocessor.prepare_test_series(df_test_local, zone, scaler_obj)
    preds_scaled = model_obj.historical_forecasts(
        test_series_scaled,
        forecast_horizon=Config.OUTPUT_CHUNK_LENGTH,
        stride=Config.OUTPUT_CHUNK_LENGTH,
        last_points_only=False, retrain=False, verbose=False
    )
    if not preds_scaled:
        raise ValueError(f"No forecasts for model: {model_name}")

    preds_scaled = concatenate(preds_scaled)
    preds_unscaled = scaler_obj.inverse_transform(preds_scaled)
    preds_df = preds_unscaled.to_dataframe().reset_index()
    if preds_df.shape[1] < 2:
        raise ValueError(f"Unexpected prediction shape for model: {model_name}")

    preds_df = preds_df.rename(
        columns={preds_df.columns[0]: "Timestamp", preds_df.columns[1]: "Predicted"}
    )[["Timestamp", "Predicted"]]
    preds_df["Timestamp"] = pd.to_datetime(preds_df["Timestamp"])

    actual_df = df_test_local[[Config.TIME_COLUMN, zone]].copy()
    actual_df.columns = ["Timestamp", "Actual"]
    actual_df["Timestamp"] = pd.to_datetime(actual_df["Timestamp"])
    merged = pd.merge(actual_df, preds_df, on="Timestamp", how="inner").dropna()
    if merged.empty:
        raise ValueError(f"No overlapping timestamps for model: {model_name}")
    return merged


def plot_predictions_for_models(
    selected_models, zone=None, start_date=None, end_date=None, months=None, max_points=4000
):
    if zone is None: zone = Config.TRAINING_ZONE

    if isinstance(selected_models, str):
        key = selected_models.strip().upper()
        if key == "ALL":
            selected_models = list_available_models(zone=zone)
        elif key == "TOP5":
            selected_models = get_top_models_by_test_mape(zone=zone, top_n=5)
        else:
            selected_models = [selected_models]

    if not selected_models:
        print("No models selected."); return None

    print("=" * 80)
    print(f"Selected N-HiTS models ({len(selected_models)}):")
    for i, m in enumerate(selected_models, 1):
        print(f"{i}. {m}")
    print("=" * 80)

    fig = go.Figure()
    metrics_rows = []
    plotted_actual = False

    for model_name in selected_models:
        try:
            df_eval_local = _prediction_df_for_model(
                model_name, zone=zone, start_date=start_date, end_date=end_date, months=months
            )
            if len(df_eval_local) > max_points:
                step = max(1, len(df_eval_local) // max_points)
                df_plot_local = df_eval_local.iloc[::step].copy()
            else:
                df_plot_local = df_eval_local.copy()

            if not plotted_actual:
                fig.add_trace(go.Scatter(
                    x=df_plot_local["Timestamp"], y=df_plot_local["Actual"],
                    mode="lines", name="Actual", line=dict(color="white", width=2)
                ))
                plotted_actual = True

            fig.add_trace(go.Scatter(
                x=df_plot_local["Timestamp"], y=df_plot_local["Predicted"],
                mode="lines", name=f"Predicted - {model_name}", line=dict(width=1.7)
            ))

            m = ModelEvaluator.compute_metrics(
                df_eval_local["Actual"].values, df_eval_local["Predicted"].values, model_name
            )
            metrics_rows.append({
                "model_name": model_name, "MAE": m["MAE"], "RMSE": m["RMSE"],
                "MAPE": m["MAPE"], "R2": m["R2"], "n_points": len(df_eval_local)
            })
            print(f"OK: {model_name} | MAPE={m['MAPE']:.2f}% | MAE={m['MAE']:.2f} | R2={m['R2']:.4f}")

        except Exception as e:
            print(f"SKIP: {model_name} | {str(e)}")

    fig.update_layout(
        title=f"Actual vs N-HiTS Predictions | Zone: {zone}",
        xaxis_title="Timestamp", yaxis_title="Demand",
        hovermode="x unified", height=550
    )
    fig.show()

    if metrics_rows:
        metrics_df = pd.DataFrame(metrics_rows).sort_values("MAPE", ascending=True)
        print("\nModel comparison:")
        print(metrics_df.to_string(index=False))
        return metrics_df

    print("No model could be plotted.")
    return None


# ============================================================================
# EXAMPLES
# ============================================================================

# 1) Plot one model:
# plot_predictions_for_models(["nhits_TPCODL_months_1_11_12_2l_512w_ctx7d_..."])

# 2) Plot all N-HiTS models for current zone:
# plot_predictions_for_models("ALL")

# 3) Plot top 5 N-HiTS models:
plot_predictions_for_models("TOP5")

print("\nReady. Use plot_predictions_for_models(...) with your model selection.")